In [1]:
!pip install -q ultralytics opencv-python-headless ipywidgets

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="SlBxRabVfF2QYLeZwq44")
project = rf.workspace("fathinahs-workspace").project("guava-seg-hwznh")
version = project.version(1)
dataset = version.download("yolo26")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 16.3 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to guava-seg-1 in yolo26:: 100%|██████████| 1692/1692 [00:00<00:00, 4262.23it/s]


In [8]:
from google.colab import files
import os
import zipfile
import glob

# Folder kerja
BASE_DIR = "/content/drive/MyDrive/ggf_guava_project"


FRAME_DIR = project.search(
    limit=10,
    in_dataset=True,
)

model_path = os.path.join(BASE_DIR, 'weights.pt')

if model_path is None:
    raise FileNotFoundError("File model .pt tidak ditemukan.")

print("Model:", model_path)
print("Upload dan ekstraksi selesai.")

Model: /content/drive/MyDrive/ggf_guava_project/weights.pt
Upload dan ekstraksi selesai.


In [12]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

from ultralytics import YOLO

# Load model
model = YOLO(model_path)

search_path = os.path.join(dataset.location, '**/*.jpg')
image_paths = glob.glob(search_path, recursive=True)

# Urutkan berdasarkan nama file
image_paths = sorted(image_paths)[:10]

if len(image_paths) == 0:
    raise FileNotFoundError(f"Tidak ada gambar yang ditemukan di {dataset.location}.")

print(f"Jumlah frame yang akan diproses: {len(image_paths)}")

for path in image_paths:
    print(os.path.basename(path))

Jumlah frame yang akan diproses: 10
20261117_20261117115107_20261117115714_115106_mp4-0009_jpg.rf.b7428bc48d247c9da1e4036de53df9c3.jpg
20261117_20261117115107_20261117115714_115106_mp4-0040_jpg.rf.0704f8f0a6c9be1b02eeb15c1a131b6a.jpg
20261117_20261117115107_20261117115714_115106_mp4-0047_jpg.rf.f1d1cac4be03925734f76ba1e8c1b8f8.jpg
20261117_20261117115107_20261117115714_115106_mp4-0054_jpg.rf.7f9b044540c18a52a419e8470fc74d3d.jpg
20261117_20261117115107_20261117115714_115106_mp4-0073_jpg.rf.d4b3cd843fc65f3d17962ba39805260c.jpg
20261117_20261117115107_20261117115714_115106_mp4-0088_jpg.rf.13e2c7ae8c3f1c2ea2af5efff5053254.jpg
20261117_20261117115107_20261117115714_115106_mp4-0101_jpg.rf.e65dc41ffb99b0cf7a22f48d92278900.jpg
20261117_20261117115107_20261117115714_115106_mp4-0118_jpg.rf.40f66a7bde7ad1c76612446a4c7bf86f.jpg
20261117_20261117115107_20261117115714_115106_mp4-0124_jpg.rf.6ba3ca9fc8d27f3427535973055facb9.jpg
20261117_20261117115107_20261117115714_115106_mp4-0127_jpg.rf.5b5581654bb

In [13]:
import ipywidgets as widgets
from IPython.display import display

# Pengaturan threshold
h_max_slider = widgets.IntSlider(
    value=25, min=0, max=179,
    description="H max:",
    continuous_update=False
)

s_min_slider = widgets.IntSlider(
    value=0, min=0, max=255,
    description="S min:",
    continuous_update=False
)

v_max_slider = widgets.IntSlider(
    value=190, min=0, max=255,
    description="V max:",
    continuous_update=False
)

conf_slider = widgets.FloatSlider(
    value=0.50, min=0.10, max=0.95, step=0.05,
    description="Confidence:",
    continuous_update=False
)

display(h_max_slider, s_min_slider, v_max_slider, conf_slider)

IntSlider(value=25, continuous_update=False, description='H max:', max=179)

IntSlider(value=0, continuous_update=False, description='S min:', max=255)

IntSlider(value=190, continuous_update=False, description='V max:', max=255)

FloatSlider(value=0.5, continuous_update=False, description='Confidence:', max=0.95, min=0.1, step=0.05)

In [14]:
# Simpan hasil segmentasi buah agar tidak perlu inference ulang
frame_data = []

for image_path in image_paths:

    image = cv2.imread(image_path)

    if image is None:
        print(f"Gagal membaca: {image_path}")
        continue

    height, width = image.shape[:2]

    # Inference model segmentasi buah
    results = model.predict(
        source=image,
        conf=conf_slider.value,
        retina_masks=True,
        verbose=False
    )

    result = results[0]

    # Mask gabungan seluruh buah dalam frame
    fruit_mask = np.zeros((height, width), dtype=np.uint8)

    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()

        for mask in masks:
            resized_mask = cv2.resize(
                mask,
                (width, height),
                interpolation=cv2.INTER_NEAREST
            )

            fruit_mask[resized_mask > 0.5] = 255

    frame_data.append({
        "path": image_path,
        "image": image,
        "fruit_mask": fruit_mask
    })

print(f"Segmentasi selesai: {len(frame_data)} frame.")

Segmentasi selesai: 10 frame.
